# 🧠 Mental Health Intelligence Platform

## Phase 2 : Data Cleaning

### Project Overview

Real-world datasets are rarely clean.

Before performing exploratory analysis or building predictive models, the dataset must be assessed for quality issues and cleaned systematically.

This notebook focuses exclusively on improving the quality of the unified OSMI Mental Health Survey dataset.

---

### Objectives

- Remove duplicate records

- Handle missing values

- Standardize categorical variables

- Clean inconsistent text

- Validate numerical variables

- Prepare a trustworthy dataset for analysis

> No feature engineering or machine learning is performed in this notebook.

## Import Required Libraries

The following libraries are required for cleaning and preprocessing the dataset.

In [135]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries Imported Successfully")

Libraries Imported Successfully


## Load Unified Dataset

Load the unified survey dataset generated during the data integration stage.

In [136]:
df = pd.read_csv("../data/processed/unified_osmi_data.csv")

print(f"Rows : {df.shape[0]:,}")

print(f"Columns : {df.shape[1]}")

Rows : 3,095
Columns : 85


## Create Working Copy

To preserve the integrity of the original dataset,
all preprocessing operations are performed on a copy.

This allows the raw dataset to remain unchanged
throughout the project.

In [137]:
clean_df = df.copy()

### Initial Data Quality Audit

Before modifying the dataset, it is essential to evaluate
its overall quality.

This audit summarizes:

- Dataset dimensions
- Missing values
- Duplicate records
- Memory usage
- Feature types

These metrics establish a baseline against which all
cleaning operations can later be evaluated.

In [139]:
print(f"Rows                : {clean_df.shape[0]:,}")

print(f"Columns             : {clean_df.shape[1]}")

print(f"Duplicate Rows      : {clean_df.duplicated().sum()}")

print(f"Total Missing       : {clean_df.isnull().sum().sum():,}")

print(f"Memory Usage (MB)   : {clean_df.memory_usage(deep=True).sum()/1024**2:.2f}")

Rows                : 3,095
Columns             : 85
Duplicate Rows      : 13
Total Missing       : 151,776
Memory Usage (MB)   : 8.58


In [140]:
audit = pd.DataFrame({

    "Data Type": clean_df.dtypes,

    "Missing": clean_df.isnull().sum(),

    "Missing %": (clean_df.isnull().sum()/len(clean_df))*100,

    "Unique Values": clean_df.nunique()

})

audit.sort_values("Missing %",ascending=False)

,Data Type,Missing,Missing %,Unique Values
disorder_ptsd,float64,3095,100.000000,0
disorder_ocd,float64,3095,100.000000,0
disorder_personality,float64,3095,100.000000,0
disorder_adh_add,float64,3095,100.000000,0
disorder_eating,float64,3095,100.000000,0
disorder_dissociative,float64,3095,100.000000,0
disorder_other,float64,3095,100.000000,0
bring_up_physical_why,float64,3095,100.000000,0
supportive_response_desc,float64,3095,100.000000,0
disorder_addictive,float64,3095,100.000000,0


### Duplicate Record Analysis

Duplicate observations can bias statistical analysis and
machine learning models by overrepresenting certain responses.

This section identifies duplicate records before deciding
whether removal is appropriate.

In [141]:
duplicates = clean_df.duplicated()

print(f"Duplicate Rows : {duplicates.sum()}")

Duplicate Rows : 13


In [142]:
clean_df[duplicates]

,source_file,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,anonymity,leave_ease,comfort_physical_mental,comfort_supervisor,discussed_with_employer,comfort_coworkers,discussed_with_coworkers,coworker_discussed_with_you,employer_importance_physical,employer_importance_mental,coverage_mental_health,know_resources,reveal_to_clients,reveal_to_coworkers,productivity_affected,productivity_percent_affected,have_prev_employers,prev_tech_company,prev_benefits,prev_know_options,prev_formal_discussion,prev_resources,prev_anonymity,prev_comfort_physical_mental,prev_comfort_supervisor,prev_discussed_with_employer,prev_comfort_coworkers,prev_discussed_with_coworkers,prev_employer_importance_physical,prev_employer_importance_mental,currently_disordered,diagnosed_disorder,disorder_anxiety,disorder_mood,disorder_psychotic,disorder_eating,disorder_adh_add,disorder_personality,disorder_ocd,disorder_ptsd,disorder_stress_response,disorder_dissociative,disorder_substance_use,disorder_addictive,disorder_other,past_disorder,sought_treatment,family_history,interference_when_treated,interference_when_not_treated,observations_less_likely_reveal,share_with_friends_family,bring_up_physical_interview,bring_up_physical_why,bring_up_mental_interview,bring_up_mental_why,openly_identified,identified_affected_career,identified_affected_how,team_reaction_if_knew,observed_unsupportive_response,unsupportive_response_desc,observed_supportive_response,supportive_response_desc,industry_support_rating,improvement_suggestions,additional_comments,willing_to_interview,age,gender,country_live,state_live,race,country_work,state_work
308,2014,2014,No,26-100,Yes,NaN,Don't know,Not sure,No,Don't know,Don't know,Don't know,Don't know,Yes,No,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
678,2014,2014,Yes,1-5,Yes,NaN,No,Yes,No,No,Yes,Very easy,Yes,Yes,No,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
821,2014,2014,Yes,1-5,Yes,NaN,No,Yes,Yes,No,Don't know,Somewhat easy,Yes,Some of them,No,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,Yes,Often,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
860,2014,2014,No,6-25,No,NaN,No,No,No,No,No,Don't know,No,No,Yes,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,Yes,Rarely,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
918,2014,2014,No,More than 1000,Yes,NaN,Yes,Yes,Yes,Yes,Yes,Very easy,Yes,Yes,No,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
959,2014,2014,No,More than 1000,Yes,NaN,Don't know,Not sure,Don't know,Don't know,Don't know,Don't know,Don't know,Yes,No,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,NaN,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1039,2014,2014,No,6-25,Yes,NaN,No,No,No,No,Don't know,Don't know,D

In [143]:
clean_df.drop_duplicates(inplace=True)

print(f"Remaining Duplicates : {clean_df.duplicated().sum()}")

Remaining Duplicates : 0


### Handle Columns with Missing Values

Some columns contain only missing values and should be dropped.

In [145]:
missing = pd.DataFrame({

    "Missing Values": clean_df.isnull().sum(),

    "Missing Percentage": (
        clean_df.isnull().sum()/len(clean_df)
    )*100

})

missing = missing.sort_values(
    "Missing Percentage",
    ascending=False
)

missing.head(25)

,Missing Values,Missing Percentage
disorder_ptsd,3082,100.000000
disorder_ocd,3082,100.000000
disorder_personality,3082,100.000000
disorder_adh_add,3082,100.000000
disorder_eating,3082,100.000000
disorder_dissociative,3082,100.000000
disorder_other,3082,100.000000
bring_up_physical_why,3082,100.000000
supportive_response_desc,3082,100.000000
disorder_addictive,3082,100.000000


In [146]:
low_missing = []
moderate_missing = []
high_missing = []

for col in clean_df.columns:

    pct = clean_df[col].isnull().mean()*100

    if pct < 10:

        low_missing.append(col)

    elif pct < 40:

        moderate_missing.append(col)

    else:

        high_missing.append(col)

print(f"Low Missing (<10%)       : {len(low_missing)}")

print(f"Moderate Missing (10-40%): {len(moderate_missing)}")

print(f"High Missing (>40%)      : {len(high_missing)}")

Low Missing (<10%)       : 14
Moderate Missing (10-40%): 4
High Missing (>40%)      : 67


### Remove Empty Features

Columns containing no information provide no analytical value and unnecessarily increase dataset complexity.

Any feature consisting entirely of missing values will therefore be removed.

In [149]:
empty_columns = clean_df.columns[
    clean_df.isnull().all()
]

print(f"Columns with 100% missing values : {len(empty_columns)}")

empty_columns

Columns with 100% missing values : 15


Index(['disorder_anxiety', 'disorder_mood', 'disorder_psychotic',
       'disorder_eating', 'disorder_adh_add', 'disorder_personality',
       'disorder_ocd', 'disorder_ptsd', 'disorder_stress_response',
       'disorder_dissociative', 'disorder_substance_use', 'disorder_addictive',
       'disorder_other', 'bring_up_physical_why', 'supportive_response_desc'],
      dtype='str')

In [150]:
clean_df.drop(
    columns=empty_columns,
    inplace=True
)

print(clean_df.shape)

(3082, 70)


### Evaluate High-Missing Features

High missingness does not necessarily imply low value.

Some survey questions were introduced only in later years and remain highly informative despite limited availability.

Each feature is evaluated individually before removal.

In [151]:
high_missing_df = missing[
    missing["Missing Percentage"] > 70
]

high_missing_df

,Missing Values,Missing Percentage
disorder_ptsd,3082,100.000000
disorder_ocd,3082,100.000000
disorder_personality,3082,100.000000
disorder_adh_add,3082,100.000000
disorder_eating,3082,100.000000
disorder_dissociative,3082,100.000000
disorder_other,3082,100.000000
bring_up_physical_why,3082,100.000000
supportive_response_desc,3082,100.000000
disorder_addictive,3082,100.000000


In [152]:
review = pd.DataFrame({

    "Column": high_missing_df.index,

    "Missing %": high_missing_df["Missing Percentage"].values,

    "Decision": "",

    "Reason": ""

})

review

,Column,Missing %,Decision,Reason
0,disorder_ptsd,100.000000,,
1,disorder_ocd,100.000000,,
2,disorder_personality,100.000000,,
3,disorder_adh_add,100.000000,,
4,disorder_eating,100.000000,,
5,disorder_dissociative,100.000000,,
6,disorder_other,100.000000,,
7,bring_up_physical_why,100.000000,,
8,supportive_response_desc,100.000000,,
9,disorder_addictive,100.000000,,


### Numerical Feature Validation

Numerical variables are inspected for unrealistic or invalid values.

Rather than immediately modifying the data, suspicious observations are first identified and reviewed.

In [153]:
numeric_cols = clean_df.select_dtypes(
    include="number"
).columns

numeric_cols

Index(['source_file', 'survey_year', 'employer_importance_physical',
       'employer_importance_mental', 'have_prev_employers',
       'prev_employer_importance_physical', 'prev_employer_importance_mental',
       'share_with_friends_family', 'identified_affected_how',
       'team_reaction_if_knew', 'industry_support_rating',
       'willing_to_interview', 'age'],
      dtype='str')

In [154]:
clean_df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
source_file,3082.0,2016.496106,2.276879,2014.0,2014.0,2017.0,2018.00,2021.0
survey_year,3082.0,2016.496106,2.276879,2014.0,2014.0,2017.0,2018.00,2021.0
employer_importance_physical,1577.0,6.317058,2.279832,0.0,5.0,7.0,8.00,10.0
employer_importance_mental,1577.0,5.027901,2.500859,0.0,3.0,5.0,7.00,10.0
have_prev_employers,1173.0,0.878090,0.327321,0.0,1.0,1.0,1.00,1.0
prev_employer_importance_physical,1546.0,5.413325,2.603817,0.0,4.0,5.0,7.00,10.0
prev_employer_importance_mental,1546.0,3.518111,2.534422,0.0,1.0,3.0,5.00,10.0
share_with_friends_family,1836.0,6.404139,2.747714,0.0,5.0,7.0,8.25,10.0
identified_affected_how,82.0,3.902439,2.502393,0.0,2.0,3.0,5.75,10.0
team_reaction_if_knew,417.0,5.354916,2.296251,0.0,4.0,5.0,7.00,10.0


In [155]:
clean_df["age"].describe()

count    1834.000000
mean       34.811341
std         9.723979
min         0.000000
25%        28.000000
50%        34.000000
75%        40.000000
max       223.000000
Name: age, dtype: float64

In [156]:
clean_df[
    (clean_df["age"]<15)|
    (clean_df["age"]>90)
]

,source_file,survey_year,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,anonymity,leave_ease,comfort_physical_mental,comfort_supervisor,discussed_with_employer,comfort_coworkers,discussed_with_coworkers,coworker_discussed_with_you,employer_importance_physical,employer_importance_mental,coverage_mental_health,know_resources,reveal_to_clients,reveal_to_coworkers,productivity_affected,productivity_percent_affected,have_prev_employers,prev_tech_company,prev_benefits,prev_know_options,prev_formal_discussion,prev_resources,prev_anonymity,prev_comfort_physical_mental,prev_comfort_supervisor,prev_discussed_with_employer,prev_comfort_coworkers,prev_discussed_with_coworkers,prev_employer_importance_physical,prev_employer_importance_mental,currently_disordered,diagnosed_disorder,past_disorder,sought_treatment,family_history,interference_when_treated,interference_when_not_treated,observations_less_likely_reveal,share_with_friends_family,bring_up_physical_interview,bring_up_mental_interview,bring_up_mental_why,openly_identified,identified_affected_career,identified_affected_how,team_reaction_if_knew,observed_unsupportive_response,unsupportive_response_desc,observed_supportive_response,industry_support_rating,improvement_suggestions,additional_comments,willing_to_interview,age,gender,country_live,state_live,race,country_work,state_work
2546,2019,2019,NaN,500-1000,False,True,I don't know,No,Yes,I don't know,I don't know,Very easy,Same level of comfort for each,Yes,False,Maybe,False,False,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,I don't know,"No, I only became aware later",None did,None did,I don't know,Same level of comfort for each,"No, none of my previous supervisors",False,At some of my previous employers,False,3.0,3.0,NaN,NaN,No,False,No,NaN,NaN,No,6.0,No,NaN,I would be concerned that would be a check mar...,False,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,0.0,male,NaN,NaN,White,NaN,NaN
2789,2020,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,"No, I don't know any",Not applicable to me,"No, because it doesn't matter",Unsure,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,1,No,NaN,NaN,Maybe,1.0,Maybe,NaN,gryry,1,1.0,1.0,NaN,NaN,thyu,NaN,1.0,thyt,ryry,NaN,5.0,f,NaN,NaN,Asian,NaN,NaN
2790,2020,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,"Yes, I know several","Yes, always","Yes, always",Yes,1-25%,NaN,1.0,"Yes, they all did","Yes, I was aware of all of them","Yes, they all did","Yes, they all did","Yes, always",Physical health,"Yes, all of my previous supervisors",1.0,"Yes, at all of my previous employers",1.0,0.0,0.0,NaN,NaN,Yes,1,Yes,NaN,NaN,Yes,0.0,Yes,NaN,NaN,1,1.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2998,2021,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,"No, I don't know any","No, because it doesn't matter","No, because it would impact me negatively",No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Don't Know,0,No,NaN,NaN,NaN,4.0,Maybe,NaN,zxzxxxz,0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,dsdsd,dsd,NaN,11.0,male,NaN,NaN,NaN,NaN,NaN
3002,2021,2021,NaN,1-5,0.0,1.0,Yes,Yes,Yes,Yes,Yes,Very easy,Physical health,Yes,1.0,Yes,NaN,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,Some did,"No, I only became aware later",None did,"Yes, they all did",Sometimes,Same level of comfort for each,"Yes, all of my previous supervisors",1.0,"Yes, at all of my previous employers",NaN,3.0,2.0,NaN,NaN,Yes,1,Yes,NaN,NaN,NaN,1.0,Yes,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,223.0,NaN,NaN,NaN,White,NaN,NaN


In [157]:
clean_df.loc[
    clean_df["age"] < 15,
    "age"
] = np.nan

clean_df.loc[
    clean_df["age"] > 90,
    "age"
] = np.nan

clean_df["age"].fillna(
    clean_df["age"].median(),
    inplace=True
)

0       34.0
1       34.0
2       34.0
3       34.0
4       34.0
        ... 
3090    33.0
3091    49.0
3092    28.0
3093    26.0
3094    38.0
Name: age, Length: 3082, dtype: float64

### Clean Categorical Features

Survey responses collected over multiple years often contain inconsistencies such as:

- Leading and trailing spaces
- Mixed capitalization
- Empty strings
- Special characters

Before standardizing categories, all textual variables are cleaned to ensure consistency.

This preprocessing improves grouping, filtering, visualization, and machine learning performance.

In [159]:
categorical_cols = clean_df.select_dtypes(include="object").columns

print(f"Categorical Columns : {len(categorical_cols)}")

Categorical Columns : 57


In [160]:
for col in categorical_cols:

    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan})
    )

In [161]:
clean_df[categorical_cols].head()

,self_employed,no_employees,tech_company,primary_role_tech,benefits,know_options,formal_discussion,resources,anonymity,leave_ease,comfort_physical_mental,comfort_supervisor,discussed_with_employer,comfort_coworkers,discussed_with_coworkers,coworker_discussed_with_you,coverage_mental_health,know_resources,reveal_to_clients,reveal_to_coworkers,productivity_affected,productivity_percent_affected,prev_tech_company,prev_benefits,prev_know_options,prev_formal_discussion,prev_resources,prev_anonymity,prev_comfort_physical_mental,prev_comfort_supervisor,prev_discussed_with_employer,prev_comfort_coworkers,prev_discussed_with_coworkers,currently_disordered,diagnosed_disorder,past_disorder,sought_treatment,family_history,interference_when_treated,interference_when_not_treated,observations_less_likely_reveal,bring_up_physical_interview,bring_up_mental_interview,bring_up_mental_why,openly_identified,identified_affected_career,observed_unsupportive_response,unsupportive_response_desc,observed_supportive_response,improvement_suggestions,additional_comments,gender,country_live,state_live,race,country_work,state_work
0,NaN,6-25,Yes,NaN,Yes,Not sure,No,Yes,Yes,Somewhat easy,Yes,Yes,No,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,No,Often,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,More than 1000,No,NaN,Don't know,No,Don't know,Don't know,Don't know,Don't know,Don't know,No,Maybe,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,Rarely,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,6-25,Yes,NaN,No,No,No,No,Don't know,Somewhat difficult,No,Yes,No,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,Rarely,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,26-100,Yes,NaN,No,Yes,No,No,No,Somewhat difficult,No,No,Yes,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,Yes,Often,NaN,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,100-500,Yes,NaN,Yes,No,Don't know,Don't know,Don't know,Don't know,Don't know,Yes,No,Some of them,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,No,Never,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Gender Standardization

Gender responses are highly inconsistent due to free-text input.

Examples include:

- Male
- male
- M
- m
- Cis Male
- Male-ish
- Female
- F
- Non-binary

To improve consistency, responses are grouped into four standardized categories:

- Male
- Female
- Other
- Unknown

This simplifies demographic analysis while preserving inclusivity.

In [163]:
clean_df["gender"].value_counts(dropna=False).head(50)

gender
NaN                                                          1274
Male                                                          758
Female                                                        294
male                                                          262
female                                                        146
M                                                              66
m                                                              50
F                                                              44
f                                                              28
Woman                                                          13
woman                                                           9
Man                                                             6
Nonbinary                                                       6
cis male                                                        6
Non-binary                                                      4
Cis

In [164]:
def clean_gender(gender):

    if pd.isna(gender):
        return "Unknown"

    gender = str(gender).lower().strip()

    male = [
        "male","m","man","cis male","male-ish",
        "cis man","mail","msle","mal","maile",
        "make","male (cis)"
    ]

    female = [
        "female","f","woman","cis female",
        "cis woman","femake","female "
    ]

    if gender in male:
        return "Male"

    elif gender in female:
        return "Female"

    elif any(x in gender for x in [
        "non","trans","queer","fluid",
        "gender","agender","nb"
    ]):
        return "Other"

    return "Other"

In [165]:
clean_df["gender"] = clean_df["gender"].apply(clean_gender)

In [166]:
clean_df["gender"].value_counts()

gender
Unknown    1274
Male       1163
Female      540
Other       105
Name: count, dtype: int64

### Country Standardization

Country names may appear in different formats across survey years.

Examples include:

- USA
- United States
- United States of America
- US

These inconsistencies are standardized to improve geographical analysis.

In [169]:
clean_df["country_live"].value_counts()

country_live
United States of America    809
United Kingdom               85
Canada                       34
Germany                      29
France                       21
Netherlands                  19
India                        18
Spain                        18
Australia                    16
Switzerland                   7
Ireland                       7
Poland                        7
Japan                         5
New Zealand                   5
Italy                         5
Mexico                        5
Russia                        4
Romania                       4
Belgium                       4
South Africa                  4
Brazil                        4
Portugal                      3
Finland                       3
Sweden                        3
Serbia                        3
Argentina                     3
Greece                        3
Ukraine                       3
Norway                        3
Bangladesh                    2
Iceland                    

In [170]:
country_map = {

    "US":"United States",

    "USA":"United States",

    "United States of America":"United States",

    "U.S.":"United States",

    "UK":"United Kingdom",

    "England":"United Kingdom"

}

In [171]:
clean_df["country_live"] = (
    clean_df["country_live"]
    .replace(country_map)
)

In [172]:
clean_df["country_live"].value_counts()

country_live
United States     809
United Kingdom     85
Canada             34
Germany            29
France             21
Netherlands        19
India              18
Spain              18
Australia          16
Switzerland         7
Ireland             7
Poland              7
Japan               5
New Zealand         5
Italy               5
Mexico              5
Russia              4
Romania             4
Belgium             4
South Africa        4
Brazil              4
Portugal            3
Finland             3
Sweden              3
Serbia              3
Argentina           3
Greece              3
Ukraine             3
Norway              3
Bangladesh          2
Iceland             2
Indonesia           2
Czech Republic      2
Pakistan            2
Hungary             2
Bulgaria            2
Austria             2
Turkey              2
Colombia            2
Israel              1
Jordan              1
Croatia             1
Singapore           1
Slovakia            1
Belarus            

### Binary Variable Standardization

Several survey questions contain binary responses.

To ensure consistency, all binary variables are standardized into a common format consisting of:

- Yes
- No

This improves downstream analysis and model preprocessing.

### 5. Handle Missing Values

Different strategies for different types of columns.

#### 5.1 Categorical Columns - Mode Imputation
    
For categorical columns with moderate missingness, we'll impute with the mode.

In [174]:
binary_columns = []

for col in clean_df.columns:

    values = clean_df[col].dropna().astype(str).str.lower().unique()

    if len(values) <= 5:

        binary_columns.append(col)

binary_columns

['self_employed',
 'primary_role_tech',
 'benefits',
 'know_options',
 'formal_discussion',
 'resources',
 'anonymity',
 'comfort_supervisor',
 'comfort_coworkers',
 'discussed_with_coworkers',
 'coworker_discussed_with_you',
 'coverage_mental_health',
 'know_resources',
 'reveal_to_clients',
 'reveal_to_coworkers',
 'productivity_affected',
 'productivity_percent_affected',
 'have_prev_employers',
 'prev_tech_company',
 'prev_benefits',
 'prev_know_options',
 'prev_formal_discussion',
 'prev_resources',
 'prev_anonymity',
 'prev_comfort_physical_mental',
 'prev_comfort_supervisor',
 'prev_discussed_with_employer',
 'prev_comfort_coworkers',
 'prev_discussed_with_coworkers',
 'currently_disordered',
 'diagnosed_disorder',
 'past_disorder',
 'family_history',
 'interference_when_treated',
 'interference_when_not_treated',
 'observations_less_likely_reveal',
 'bring_up_physical_interview',
 'bring_up_mental_interview',
 'identified_affected_career',
 'observed_unsupportive_response',
 'o

In [181]:
# Define clean lists
yes_vals = ["yes", "y", "true", "1", "1.0", "sometimes"]
no_vals = ["no", "n", "false", "0", "0.0"]
dont_know_vals = ["don't know", "dont know", "not sure", "i don't know"]

def clean_binary(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    
    if value in yes_vals:
        return "Yes"
    if value in no_vals:
        return "No"
    if value in dont_know_vals or "don't" in value or "not sure" in value:
        return "Don't Know"
    if "not eligible" in value or "n/a" in value:
        return "Not Applicable"
    
    return value.title()

# Apply to all binary columns
for col in binary_columns:
    clean_df[col] = clean_df[col].apply(clean_binary)

In [182]:
for col in binary_columns[:5]:

    print("="*50)

    print(col)

    print(clean_df[col].value_counts())

self_employed
self_employed
No     1086
Yes     142
Name: count, dtype: int64
primary_role_tech
primary_role_tech
Yes    1463
No      114
Name: count, dtype: int64
benefits
benefits
Yes               970
Don't Know        654
No                508
Not Applicable     48
Name: count, dtype: int64
know_options
know_options
No            1256
Yes           1094
Don't Know     310
Name: count, dtype: int64
formal_discussion
formal_discussion
No            1780
Yes            707
Don't Know     336
Name: count, dtype: int64


### Data Type Optimization

Optimizing data types reduces memory usage and improves computational efficiency without altering the underlying information.

In [183]:
before = clean_df.memory_usage(deep=True).sum()/1024**2

print(f"Before Optimization : {before:.2f} MB")

Before Optimization : 8.49 MB


In [184]:
for col in clean_df.select_dtypes(include="object"):

    if clean_df[col].nunique() < 30:

        clean_df[col] = clean_df[col].astype("category")

In [185]:
for col in clean_df.select_dtypes(include="int64"):

    clean_df[col] = pd.to_numeric(
        clean_df[col],
        downcast="integer"
    )

for col in clean_df.select_dtypes(include="float64"):

    clean_df[col] = pd.to_numeric(
        clean_df[col],
        downcast="float"
    )

In [186]:
after = clean_df.memory_usage(deep=True).sum()/1024**2

print(f"After Optimization : {after:.2f} MB")

print(f"Reduced By : {before-after:.2f} MB")

After Optimization : 1.63 MB
Reduced By : 6.86 MB


In [188]:
quality_report = pd.DataFrame({
    "Data Type": clean_df.dtypes,
    "Missing Values": clean_df.isnull().sum(),
    "Missing Percentage": (
        clean_df.isnull().sum()/len(clean_df)
    )*100,
    "Unique Values": clean_df.nunique()
})

quality_report.sort_values(
    "Missing Percentage",
    ascending=False
)

,Data Type,Missing Values,Missing Percentage,Unique Values
identified_affected_how,float32,3000,97.339390,10
productivity_percent_affected,category,2888,93.705386,4
identified_affected_career,category,2860,92.796885,2
know_resources,category,2823,91.596366,3
reveal_to_coworkers,category,2823,91.596366,5
productivity_affected,category,2823,91.596366,4
reveal_to_clients,category,2823,91.596366,5
coverage_mental_health,category,2823,91.596366,2
unsupportive_response_desc,str,2678,86.891629,402
team_reaction_if_knew,float32,2665,86.469825,11


### Remove Low-Value Features

Following the missing value assessment, certain variables were found to contain extremely high proportions of missing values.

These variables provide limited analytical value and are therefore removed to reduce dataset complexity while preserving the integrity of the remaining information.

Only features that are either sparsely populated or not essential for the objectives of this project are removed.

In [190]:
high_missing_cols = [
    "identified_affected_how",
    "productivity_percent_affected",
    "identified_affected_career",
    "productivity_affected",
    "know_resources",
    "reveal_to_coworkers",
    "reveal_to_clients",
    "coverage_mental_health"
]

cols_to_drop = [
    col for col in high_missing_cols
    if col in clean_df.columns
]

print("Columns to Remove:")
print(cols_to_drop)

clean_df.drop(
    columns=cols_to_drop,
    inplace=True
)

print(f"Remaining Columns : {clean_df.shape[1]}")

Columns to Remove:
['identified_affected_how', 'productivity_percent_affected', 'identified_affected_career', 'productivity_affected', 'know_resources', 'reveal_to_coworkers', 'reveal_to_clients', 'coverage_mental_health']
Remaining Columns : 62


### Remove Sparse State-Level Information

State-level variables contain information only for respondents from selected countries and exhibit substantial missingness.

Because country-level analysis is sufficient for the objectives of this project, these variables are removed.

In [192]:
state_cols = [
    col
    for col in ["state_live", "state_work"]
    if col in clean_df.columns
]

clean_df.drop(
    columns=state_cols,
    inplace=True
)

print("Removed:", state_cols)

Removed: ['state_live', 'state_work']


### Standardize Race Categories

Race responses are standardized to improve consistency across the dataset.

Rare categories containing only a small number of observations are grouped into a common **Other** category to simplify analysis and reduce category sparsity.

In [194]:
if "race" in clean_df.columns:

    # Convert category -> object
    clean_df["race"] = clean_df["race"].astype("object")

    race_map = {
        "White": "White",
        "Black or African American": "Black/African American",
        "Asian": "Asian",
        "Hispanic/Latino": "Hispanic/Latino",
        "More than one race": "More than one race",
        "Prefer not to answer": "Prefer not to answer"
    }

    clean_df["race"] = clean_df["race"].replace(race_map)

    rare_races = clean_df["race"].value_counts()

    rare_races = rare_races[rare_races < 10].index

    clean_df["race"] = clean_df["race"].replace(rare_races, "Other")

    # Convert back to category
    clean_df["race"] = clean_df["race"].astype("category")

In [195]:
clean_df["race"].value_counts(dropna=False)

race
NaN                           1981
White                          956
Asian                           55
More than one of the above      37
I prefer not to answer          30
Black/African American          18
Other                            5
Name: count, dtype: int64

### Final Missing Value Treatment

Following structural cleaning, the remaining missing values are handled using appropriate imputation techniques.

The chosen strategy depends on the data type:

- Numerical Features → Median
- Categorical Features → Mode

Median imputation is preferred for numerical variables because it is more robust to outliers than the mean.

Mode imputation preserves the most frequently observed category for categorical variables.

In [196]:
categorical_cols = clean_df.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:

    if (
        clean_df[col].isnull().sum() > 0
        and
        clean_df[col].isnull().sum() < len(clean_df) * 0.5
    ):

        clean_df[col].fillna(
            clean_df[col].mode()[0],
            inplace=True
        )

In [197]:
numeric_cols = clean_df.select_dtypes(
    include=["number"]
).columns

for col in numeric_cols:

    if clean_df[col].isnull().sum() > 0:

        clean_df[col].fillna(
            clean_df[col].median(),
            inplace=True
        )

In [198]:
clean_df.isnull().sum().sort_values(
    ascending=False
).head(20)

unsupportive_response_desc        2678
team_reaction_if_knew             2665
willing_to_interview              2665
additional_comments               2611
diagnosed_disorder                2567
race                              1981
observed_unsupportive_response    1911
country_live                      1911
country_work                      1911
observed_supportive_response      1911
have_prev_employers               1909
interference_when_not_treated     1909
currently_disordered              1909
bring_up_mental_interview         1909
self_employed                     1854
improvement_suggestions           1820
bring_up_mental_why               1680
prev_discussed_with_coworkers     1544
prev_comfort_physical_mental      1536
prev_resources                    1536
dtype: int64

### Remove Free-Text Features

The unified dataset contains several free-text response fields.

Although these variables may be useful for Natural Language Processing (NLP), they are outside the scope of this structured analysis and predictive modeling project.

Therefore, these variables are removed from the modeling dataset while remaining available in the raw dataset if future text analysis is required.

In [200]:
text_cols = [
    "improvement_suggestions",
    "additional_comments",
    "bring_up_mental_why",
    "unsupportive_response_desc"
]

text_cols = [
    col
    for col in text_cols
    if col in clean_df.columns
]

clean_df.drop(
    columns=text_cols,
    inplace=True
)

print(text_cols)

['improvement_suggestions', 'additional_comments', 'bring_up_mental_why', 'unsupportive_response_desc']


### Final Dataset Validation

A final validation is performed to confirm that the cleaning process has successfully improved the overall quality of the dataset.

The following metrics are reviewed:

- Dataset dimensions
- Duplicate records
- Missing values
- Memory usage
- Data types

This validation confirms that the dataset is ready for exploratory data analysis.

In [202]:
quality_report = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Remaining Missing Values",
        "Memory Usage (MB)"
    ],
    "Value": [
        clean_df.shape[0],
        clean_df.shape[1],
        clean_df.duplicated().sum(),
        clean_df.isnull().sum().sum(),
        round(clean_df.memory_usage(deep=True).sum()/1024**2,2)
    ]
})

quality_report

,Metric,Value
0,Rows,3082.00
1,Columns,56.00
2,Duplicate Rows,0.00
3,Remaining Missing Values,67258.00
4,Memory Usage (MB),0.52


### Export Clean Dataset

The cleaned dataset is exported to the `processed` directory and will serve as the primary input for all remaining phases of the project.

No additional cleaning should be required after this point.

In [204]:
clean_df.to_csv(
    "../data/processed/cleaned_osmi_data.csv",
    index=False
)

print("✅ Clean dataset exported successfully.")

✅ Clean dataset exported successfully.
